[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/certified-journeys/certified-journeys.github.io/blob/main/courses/llama-certified/notebooks/day-06-qlora-finetuning.ipynb#scrollTo=f0a1b2c3)

---
# Day 6 · Fine-Tuning with QLoRA on Consumer Hardware
**certified-journeys / llama-certified** · Day 6 · Fine-Tuning

> **Goal for today:** Fine-tune Llama-3.2-1B on a small synthetic Alpaca-format dataset using QLoRA (4-bit quantization + LoRA), save the adapter, merge it, and compare the fine-tuned model against the base model on held-out prompts.


In [ ]:
%pip install -q \
    transformers>=4.40.0 \
    trl>=0.8.6 \
    peft>=0.10.0 \
    bitsandbytes>=0.43.0 \
    datasets>=2.18.0 \
    accelerate>=0.29.0 \
    torch


## Step 1 · Understanding QLoRA

**QLoRA** (Quantized Low-Rank Adaptation) is a fine-tuning technique that makes large model training feasible on consumer hardware by combining two ideas:

| Technique | What it does | VRAM saved |
|---|---|---|
| 4-bit NF4 quantization | Loads base model weights in 4-bit | ~75% vs fp32 |
| LoRA adapters | Trains only small rank-decomposition matrices | Trainable params < 1% of model |
| Double quantization | Quantizes the quantization constants | Additional ~0.5 GB |

### LoRA in one equation

For a weight matrix `W`, LoRA learns two low-rank matrices `A` (d×r) and `B` (r×k):
```
W_adapted = W_frozen + alpha/r * (B @ A)
```
With `r=16` and `alpha=32`, the scale factor is 2 — amplifying the adapter signal relative to the frozen weights. Only `A` and `B` are updated during training.

### Hardware requirements

| Model | r | Approx VRAM needed |
|---|---|---|
| Llama-3.2-1B | 16 | ~3 GB |
| Llama-3.2-3B | 16 | ~5 GB |
| Llama-3.1-8B | 16 | ~8 GB |
| Llama-3.1-8B | 64 | ~12 GB (OOM on 8 GB cards) |


In [ ]:
# Step 1: Generate 20 synthetic Alpaca-format training examples
# In production you would load a real dataset, e.g.:
#   from datasets import load_dataset
#   ds = load_dataset('tatsu-lab/alpaca', split='train')

import json
import random

ALPACA_TEMPLATE = (
    'Below is an instruction that describes a task. '
    'Write a response that appropriately completes the request.\n\n'
    '### Instruction:\n{instruction}\n\n'
    '### Response:\n{output}'
)

# 20 synthetic Python Q&A pairs
QA_PAIRS = [
    ('What does the `len()` function return?',
     'The `len()` function returns the number of items in an object such as a list, string, or tuple.'),
    ('How do you open a file for reading in Python?',
     'Use `open(\'filename.txt\', \'r\')` or, preferably, `with open(\'filename.txt\') as f:` for automatic resource cleanup.'),
    ('What is a Python dictionary?',
     'A dictionary is a mutable, unordered collection of key-value pairs. Keys must be hashable; values can be any type.'),
    ('How do you handle exceptions in Python?',
     'Use a try/except block: `try: ...` then `except SomeError as e: ...`. Add `finally:` for cleanup code that always runs.'),
    ('What is the difference between `==` and `is` in Python?',
     '`==` tests value equality; `is` tests object identity (same memory address). Use `is` only for singletons like None.'),
    ('How do you create a virtual environment?',
     'Run `python -m venv .venv` to create it, then `source .venv/bin/activate` (macOS/Linux) or `.venv\\Scripts\\activate` (Windows).'),
    ('What does `*args` mean in a function definition?',
     '`*args` collects extra positional arguments into a tuple. The function can then iterate over `args` to process any number of inputs.'),
    ('What is a list comprehension?',
     'A compact way to build a list: `[expr for item in iterable if condition]`. Example: `[x**2 for x in range(10) if x % 2 == 0]`.'),
    ('How do you sort a list in Python?',
     'Use `list.sort()` to sort in-place, or `sorted(list)` to return a new sorted list. Both accept a `key=` argument.'),
    ('What is a Python generator?',
     'A function that uses `yield` to produce values lazily. It pauses at each `yield` and resumes on the next `next()` call, using minimal memory.'),
    ('How do you merge two dictionaries in Python 3.9+?',
     'Use the `|` operator: `merged = dict_a | dict_b`. Values from `dict_b` overwrite duplicates from `dict_a`.'),
    ('What is the Global Interpreter Lock (GIL)?',
     'A mutex that allows only one thread to execute Python bytecode at a time. It simplifies memory management but limits CPU-bound multithreading.'),
    ('How do you reverse a list?',
     'Use `list.reverse()` in-place, `list[::-1]` for a reversed copy, or `reversed(list)` for a lazy iterator.'),
    ('What is `__init__.py` used for?',
     'It marks a directory as a Python package and can contain initialization code that runs when the package is imported.'),
    ('How do you convert a string to an integer?',
     'Use `int(\'42\')`. For floats first: `int(float(\'3.14\'))`. Raises `ValueError` if the string is not a valid number.'),
    ('What is the `@property` decorator?',
     'It turns a method into a read-only attribute accessed without parentheses. Pair with `@attr.setter` to allow writes.'),
    ('How do you check if a key exists in a dictionary?',
     'Use the `in` operator: `if \'key\' in my_dict:`. Avoid `dict.has_key()` — it was removed in Python 3.'),
    ('What is `enumerate()` used for?',
     'It adds a counter to an iterable: `for i, item in enumerate(my_list):`. Avoids manual index tracking.'),
    ('How do you write a decorator in Python?',
     'Define a function that takes another function as an argument, defines a wrapper inside it, and returns the wrapper. Use `@functools.wraps(func)` to preserve metadata.'),
    ('What is the difference between `deepcopy` and `copy`?',
     '`copy.copy()` creates a shallow copy — nested objects are shared. `copy.deepcopy()` recursively copies all nested objects, giving a fully independent clone.'),
]

def make_example(instruction: str, output: str) -> dict:
    return {'text': ALPACA_TEMPLATE.format(instruction=instruction, output=output)}

raw_data = [make_example(inst, out) for inst, out in QA_PAIRS]

# Split: 15 train, 5 eval
random.seed(42)
random.shuffle(raw_data)
train_data = raw_data[:15]
eval_data  = raw_data[15:]

print(f'Training examples : {len(train_data)}')
print(f'Eval examples     : {len(eval_data)}')
print('\nSample training example:')
print(train_data[0]['text'])


### What just happened?

- We generated 20 synthetic Python Q&A pairs in **Alpaca format** — the instruction/response structure used to fine-tune the original Alpaca model.
- Each example is a single string in `text` — `SFTTrainer` expects this field by default.
- **15 examples for training, 5 for eval** — tiny by production standards, but enough to verify the training pipeline runs correctly.
- In production, load a real dataset: `load_dataset('tatsu-lab/alpaca', split='train').select(range(500))`.


## Step 2 · Configure QLoRA and Load the Base Model

Loading the model in 4-bit requires a `BitsAndBytesConfig`. The key parameters:

| Parameter | Value | Why |
|---|---|---|
| `load_in_4bit` | True | Enable 4-bit NF4 quantization |
| `bnb_4bit_quant_type` | `'nf4'` | NormalFloat4 — best quality for LLM weights |
| `bnb_4bit_compute_dtype` | `bfloat16` | Computation stays in bf16 for numerical stability |
| `bnb_4bit_use_double_quant` | True | Saves ~0.4 bits/param extra |


In [ ]:
# Step 2: Configure BitsAndBytes for 4-bit loading and set up LoRA config

import torch
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
)
from peft import LoraConfig, TaskType, get_peft_model

# --- BitsAndBytes 4-bit config ---
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',                    # NormalFloat4 quantization
    bnb_4bit_compute_dtype=torch.bfloat16,        # compute in bf16 during forward pass
    bnb_4bit_use_double_quant=True,               # quantize the quantization constants too
)

# --- LoRA config ---
# r=16: rank of the adapter matrices. Start here; double only if model underfits.
# alpha=32: scale = alpha/r = 2. Controls adapter signal strength.
# dropout=0.05: light regularization to prevent overfitting on small datasets.
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias='none',
    # Target the attention projection matrices — these have the biggest impact on instruction following
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj'],
)

MODEL_ID = 'meta-llama/Llama-3.2-1B'
# For Colab without gated access, swap MODEL_ID for a public 1B model:
# MODEL_ID = 'Qwen/Qwen2.5-0.5B'  # public, same pipeline

print('BitsAndBytes config:')
print(f'  load_in_4bit          : {bnb_config.load_in_4bit}')
print(f'  bnb_4bit_quant_type   : {bnb_config.bnb_4bit_quant_type}')
print(f'  bnb_4bit_compute_dtype: {bnb_config.bnb_4bit_compute_dtype}')
print(f'  double_quant          : {bnb_config.bnb_4bit_use_double_quant}')
print()
print('LoRA config:')
print(f'  r              : {lora_config.r}')
print(f'  alpha          : {lora_config.lora_alpha}')
print(f'  scale (alpha/r): {lora_config.lora_alpha / lora_config.r}')
print(f'  dropout        : {lora_config.lora_dropout}')
print(f'  target_modules : {lora_config.target_modules}')

# Count trainable parameters helper (used after model load)
def count_trainable_params(model) -> dict:
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return {
        'total': total,
        'trainable': trainable,
        'pct': round(100 * trainable / total, 3) if total else 0,
    }


### What just happened?

- We defined the `BitsAndBytesConfig` — this tells the loader to quantize weights to 4-bit NF4 format on the fly.
- The `LoraConfig` targets `q_proj, k_proj, v_proj, o_proj` — the four attention projection matrices. These are the most impactful for instruction-following behavior.
- **Scale = alpha/r = 2**: at `r=16, alpha=32` the adapter signal is amplified 2x relative to its initialization. This is the standard starting point.
- `count_trainable_params` will show us that only ~0.5% of parameters are trainable — the rest stay frozen in 4-bit.


## Step 3 · Build the HuggingFace Dataset and Load Model

SFTTrainer expects a `datasets.Dataset` object. We convert our list of dicts to that format, then load the base model and tokenizer.


In [ ]:
# Step 3: Build HuggingFace Dataset objects and load model + tokenizer

from datasets import Dataset

train_ds = Dataset.from_list(train_data)
eval_ds  = Dataset.from_list(eval_data)

print(f'train_ds: {train_ds}')
print(f'eval_ds : {eval_ds}')
print()

# --- Load tokenizer ---
# Note: requires HuggingFace login for gated meta-llama models:
#   from huggingface_hub import login; login(token='hf_...')
# For a public alternative uncomment: MODEL_ID = 'Qwen/Qwen2.5-0.5B'

print(f'Loading tokenizer for {MODEL_ID} ...')
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
    padding_side='right',   # SFTTrainer needs right-padding for batch packing
)

# SFTTrainer requires a pad token; Llama 3 uses EOS as pad when unset
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f'Vocab size    : {tokenizer.vocab_size}')
print(f'Pad token     : {tokenizer.pad_token!r}')
print(f'BOS token     : {tokenizer.bos_token!r}')
print(f'EOS token     : {tokenizer.eos_token!r}')

# --- Load model in 4-bit ---
print(f'\nLoading model {MODEL_ID} in 4-bit ...')
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map='auto',          # auto-places layers across available GPU/CPU
    trust_remote_code=True,
)

# Disable KV cache during training (saves memory; re-enable for inference)
model.config.use_cache = False
model.config.pretraining_tp = 1   # disable tensor parallelism for single-GPU training

params = count_trainable_params(model)
print(f'\nBase model parameters:')
print(f'  Total      : {params["total"]:,}')
print(f'  Trainable  : {params["trainable"]:,}')
print(f'  Trainable%  : {params["pct"]}%')


### What just happened?

- `Dataset.from_list()` converts our Python list to a HuggingFace Dataset in one line.
- The tokenizer is loaded with `padding_side='right'` — required because SFTTrainer packs sequences and needs consistent padding direction.
- **`device_map='auto'`** lets Accelerate place model layers across GPU(s) and CPU automatically — essential for consumer GPUs.
- `use_cache=False` disables the KV cache during training — it's only needed for autoregressive inference.


## Step 4 · Configure SFTTrainer and Run Fine-Tuning

TRL's `SFTTrainer` wraps `Trainer` with supervised fine-tuning defaults. It handles:
- Tokenizing the `text` column
- Loss masking on padding tokens
- Sequence packing for efficiency


In [ ]:
# Step 4: Configure and run SFTTrainer with QLoRA

from trl import SFTTrainer, SFTConfig
from transformers import TrainingArguments

OUTPUT_DIR = './llama-3.2-1b-alpaca-qlora'

training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=4,   # effective batch = 2 * 4 = 8
    warmup_steps=5,
    learning_rate=2e-4,
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    logging_steps=5,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    report_to='none',               # disable wandb/tensorboard for this demo
    optim='paged_adamw_8bit',       # 8-bit paged AdamW saves optimizer state VRAM
    max_seq_length=512,
    dataset_text_field='text',      # column in our Dataset that holds the formatted string
    packing=False,                  # disable packing for simplicity on tiny dataset
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    peft_config=lora_config,        # SFTTrainer wraps the model with PEFT automatically
    tokenizer=tokenizer,
)

# Show trainable parameters after PEFT wrapping
peft_params = count_trainable_params(trainer.model)
print('After PEFT wrapping:')
print(f'  Total      : {peft_params["total"]:,}')
print(f'  Trainable  : {peft_params["trainable"]:,}')
print(f'  Trainable%  : {peft_params["pct"]}%')
print()

# Run training
print('Starting training ...')
train_result = trainer.train()

print('\nTraining complete.')
print(f'  Runtime        : {train_result.metrics["train_runtime"]:.1f}s')
print(f'  Samples/sec    : {train_result.metrics["train_samples_per_second"]:.2f}')
print(f'  Final loss     : {train_result.metrics["train_loss"]:.4f}')


### What just happened?

- `SFTTrainer` applies the `lora_config` automatically via `peft_config` — no manual `get_peft_model()` call needed.
- **`paged_adamw_8bit`** stores optimizer states in CPU RAM and pages them to GPU when needed — critical for staying within 8 GB VRAM.
- `gradient_accumulation_steps=4` simulates a larger batch size without needing more VRAM.
- After PEFT wrapping, the trainable parameter count drops to ~0.5% of total — only the LoRA `A` and `B` matrices update.


## Step 5 · Save the LoRA Adapter

After training we save only the adapter weights — not the full model. The adapter directory is ~10–50 MB, not 2 GB.


In [ ]:
# Step 5: Save adapter and verify the saved files

import os

ADAPTER_DIR = './llama-3.2-1b-alpaca-adapter'

# Save only the LoRA adapter weights + config
trainer.model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)

print(f'Adapter saved to: {ADAPTER_DIR}')
print()
print('Files in adapter directory:')
for fname in sorted(os.listdir(ADAPTER_DIR)):
    fpath = os.path.join(ADAPTER_DIR, fname)
    size_kb = os.path.getsize(fpath) / 1024
    print(f'  {fname:<40} {size_kb:>8.1f} KB')

# The key files to verify:
required_files = ['adapter_config.json', 'adapter_model.safetensors']
for fname in required_files:
    exists = os.path.exists(os.path.join(ADAPTER_DIR, fname))
    status = 'PRESENT' if exists else 'MISSING'
    print(f'  {fname}: {status}')


### What just happened?

- `save_pretrained` on a PEFT model saves **only the adapter** — `adapter_config.json` + `adapter_model.safetensors`.
- `adapter_config.json` stores `r`, `alpha`, `target_modules`, and `base_model_name_or_path` so the adapter can be reloaded without knowing those values.
- **The adapter is ~10–50 MB** for a 1B model with `r=16`. The base model (not saved) is ~0.5 GB in 4-bit.
- To share the adapter publicly: push to HuggingFace Hub with `trainer.model.push_to_hub('username/model-name')`.


## Step 6 · Load the Adapter and Run Inference

To use the fine-tuned model: load the base model, then attach the adapter with `PeftModel.from_pretrained`. This is the lightweight deployment path — you do not need to merge.


In [ ]:
# Step 6: Load adapter from disk and generate a response

from peft import PeftModel

# Re-load the base model in 4-bit (in a real workflow this stays in memory)
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map='auto',
    trust_remote_code=True,
)
base_model.config.use_cache = True   # re-enable KV cache for inference

# Attach the LoRA adapter on top of the frozen base
ft_model = PeftModel.from_pretrained(base_model, ADAPTER_DIR)
ft_model.eval()  # disable dropout for inference

inference_tokenizer = AutoTokenizer.from_pretrained(ADAPTER_DIR)
if inference_tokenizer.pad_token is None:
    inference_tokenizer.pad_token = inference_tokenizer.eos_token

def generate_response(model, tokenizer, prompt: str, max_new_tokens: int = 150) -> str:
    """Run greedy inference and return the generated text (excluding prompt)."""
    inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,          # greedy for reproducibility
            temperature=1.0,
            pad_token_id=tokenizer.eos_token_id,
        )
    # Decode only the newly generated tokens (exclude the prompt)
    new_tokens = outputs[0][inputs['input_ids'].shape[-1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True)


TEST_PROMPT = (
    'Below is an instruction that describes a task. '
    'Write a response that appropriately completes the request.\n\n'
    '### Instruction:\nWhat is a Python decorator?\n\n'
    '### Response:\n'
)

ft_response = generate_response(ft_model, inference_tokenizer, TEST_PROMPT)
print('=== Fine-tuned model response ===')
print(ft_response)


### What just happened?

- `PeftModel.from_pretrained(base_model, ADAPTER_DIR)` layers the adapter on top of the frozen 4-bit base — no merging required.
- `use_cache=True` is re-enabled for inference — it speeds up token-by-token generation by caching attention key-values.
- We slice `outputs[0][input_length:]` to return **only the generated tokens**, not the prompt echo.
- `do_sample=False` (greedy) gives deterministic outputs — useful for evaluation and regression testing.


## Step 7 · Merge Adapter into Base Model

Merging bakes the adapter weights into the base model permanently. The result is a standard model — no PEFT dependency, faster inference (no adapter overhead), easy to export to GGUF.


In [ ]:
# Step 7: Merge adapter into base model and save the merged model

MERGED_DIR = './llama-3.2-1b-alpaca-merged'

# merge_and_unload() performs W_adapted = W_frozen + scale * (B @ A)
# then removes the PEFT overhead, returning a standard HF model
merged_model = ft_model.merge_and_unload()

# Save in safetensors format (safer than pytorch pickle)
merged_model.save_pretrained(MERGED_DIR, safe_serialization=True)
inference_tokenizer.save_pretrained(MERGED_DIR)

print(f'Merged model saved to: {MERGED_DIR}')
print()
print('Files in merged directory:')
for fname in sorted(os.listdir(MERGED_DIR)):
    fpath = os.path.join(MERGED_DIR, fname)
    size_mb = os.path.getsize(fpath) / (1024 ** 2)
    print(f'  {fname:<45} {size_mb:>8.1f} MB')

# Verify the merged model has no adapter_config.json (it's a plain model now)
has_adapter_config = os.path.exists(os.path.join(MERGED_DIR, 'adapter_config.json'))
print(f'\nHas adapter_config.json (should be False): {has_adapter_config}')
print('Merge complete — this is a standard HuggingFace model, no PEFT dependency.')


### What just happened?

- `merge_and_unload()` computes `W + scale*(B@A)` for every targeted layer and stores the result in the base weight matrices.
- The merged model has **no `adapter_config.json`** — it is a plain `AutoModelForCausalLM` with no PEFT dependency.
- `safe_serialization=True` saves in safetensors format — resistant to pickle exploits and supports memory-mapped loading.
- **Trade-off**: merged model is larger (full precision) but faster at inference and easier to deploy or convert to GGUF.


## Step 8 · Compare Base vs Fine-Tuned on 5 Held-Out Prompts

Load the merged model and run both it and the base model on the 5 eval examples to see the effect of fine-tuning.


In [ ]:
# Step 8: Side-by-side comparison — base model vs fine-tuned on held-out prompts

# Load the merged fine-tuned model
from transformers import pipeline

ft_pipe = pipeline(
    'text-generation',
    model=MERGED_DIR,
    tokenizer=MERGED_DIR,
    torch_dtype=torch.bfloat16,
    device_map='auto',
    max_new_tokens=120,
    do_sample=False,
)

# Use the 5 eval examples (already in Alpaca format in eval_data)
# Extract instruction and build the prompt prefix (without the answer)
def extract_instruction_prompt(text: str) -> str:
    """Extract the prompt part of an Alpaca example (up to ### Response:)."""
    resp_idx = text.find('### Response:\n')
    if resp_idx == -1:
        return text
    return text[:resp_idx + len('### Response:\n')]

def extract_reference(text: str) -> str:
    """Extract the ground-truth response from an Alpaca example."""
    marker = '### Response:\n'
    idx = text.find(marker)
    return text[idx + len(marker):].strip() if idx != -1 else ''


print('=== Base vs Fine-tuned: 5 held-out examples ===\n')

for i, example in enumerate(eval_data):
    prompt    = extract_instruction_prompt(example['text'])
    reference = extract_reference(example['text'])

    ft_out = ft_pipe(prompt)[0]['generated_text']
    # Isolate only the newly generated part
    ft_answer = ft_out[len(prompt):].strip()

    print(f'--- Example {i+1} ---')
    # Show just the instruction line
    instr_line = [line for line in prompt.split('\n') if line.startswith('###')]
    print(f'Instruction : {instr_line[0].replace("### Instruction:", "").strip()}')
    print(f'Reference   : {reference[:120]}...' if len(reference) > 120 else f'Reference   : {reference}')
    print(f'FT model    : {ft_answer[:120]}...' if len(ft_answer) > 120 else f'FT model    : {ft_answer}')
    print()


### What just happened?

- We used `pipeline()` on the **merged model directory** — standard HuggingFace inference, no PEFT import needed.
- The eval examples are the 5 held-out Q&A pairs the model was not trained on.
- With only 15 training examples and 3 epochs, the fine-tuned model should produce more Alpaca-style answers, but gains may be subtle at this scale.
- **Interpretation**: If FT answers closely match the reference format (concise, structured), the adapter is working. For production, evaluate on 500+ held-out examples with a metric like ROUGE or BERTScore.


In [ ]:
# Challenge: Implement a LoRA hyperparameter sweep
#
# Write a function `sweep_lora` that trains the model with three different
# (r, alpha) pairs and returns the final eval loss for each:
#
#   configs = [(8, 16), (16, 32), (32, 64)]
#
# For each config:
#   1. Build a new LoraConfig with the given r and alpha
#   2. Train for 1 epoch only (to keep runtime short)
#   3. Record trainer.evaluate()['eval_loss']
#   4. Clean up / delete the PEFT model before the next config
#
# Return a list of dicts: [{'r': 8, 'alpha': 16, 'eval_loss': ...}, ...]
#
# Then print the config with the lowest eval_loss.
#
# Scaffold:

def sweep_lora(
    base_model_id: str,
    train_dataset,
    eval_dataset,
    tokenizer,
    bnb_config,
    configs: list[tuple[int, int]],
) -> list[dict]:
    results = []
    for r, alpha in configs:
        # TODO: build LoraConfig, load model, train 1 epoch, record eval_loss
        pass
    return results

# Uncomment to run (requires sufficient VRAM and ~10 min runtime):
# results = sweep_lora(MODEL_ID, train_ds, eval_ds, tokenizer, bnb_config, [(8,16),(16,32),(32,64)])
# best = min(results, key=lambda x: x['eval_loss'])
# print(f'Best config: r={best["r"]}, alpha={best["alpha"]}, eval_loss={best["eval_loss"]:.4f}')


---
## Day 6 key concepts recap

| Concept | What to remember |
|---|---|
| QLoRA | 4-bit NF4 quantization + LoRA = fine-tune a 7B on 8 GB VRAM |
| r and alpha | Start at r=16, alpha=32 (scale=2). Double r only if underfitting after 3 epochs |
| target_modules | q_proj, k_proj, v_proj, o_proj are the go-to for instruction following |
| paged_adamw_8bit | Pages optimizer states to CPU RAM — saves 2-4 GB VRAM vs Adam |
| Adapter vs merged | Adapter = ~50 MB, requires PEFT at inference. Merged = full model, no PEFT dep |
| use_cache | Disable during training, re-enable for inference |
| Eval strategy | Always hold out examples; loss alone does not tell you if outputs improved |

> **Tip:** Use r=16, alpha=32, dropout=0.05 as your starting LoRA config. Double r only if the model underfits after 3 epochs. Increasing r increases VRAM — on 8 GB cards, r=64 will OOM on a 7B model.

---
## What's next
**Day 7** → GGUF Conversion and llama.cpp — convert your merged model to GGUF format for CPU inference with llama.cpp and Ollama.

Mark Day 6 complete in your [tracker](../index.html).
